# Extra: rs_100 "hall" model (CLAMPbase, full K=1728, 100% coverage, seed1)

**Environment:** `clamp-analyses`

CLAMPbase counterpart to `00_bp_saturation_study_ora_rs_100.ipynb` (the
CLAMPfull anchor). Single, manually-added anchor point for the saturation
curve in `01_bp_saturation_study_plot.ipynb`. Not part of the
auto-discovered `08_saturation_study` grid (which only sweeps K up to 25%
study coverage) — instead it uses the already-built `CLAMPbase` model at
100% **sample** coverage from `06_bp_coverage_rshall/06_bp_coverage_hall_rs_100`,
which has the maximum K (1728 LVs) and serves as the natural full-K anchor
point for the saturation plot's pre-wired "100%" styling.

CLAMPbase only, seed1 only, single-threaded (no `BiocParallel`/`mclapply`)
since this is just one model.

`pvalueCutoff = 0.05` (not `1`) — this combo (K=1728, MSigDB, no p-value
filtering) OOM-killed two earlier local attempts at ~80GB RSS, because
`enricher()` returns every one of 35k MSigDB terms per LV (each with a
verbose `geneID` column) with no filtering. `0.05` is the loosest FDR
threshold used downstream and is behavior-preserving for the coverage
calculation (BH adjustment never decreases a p-value, so anything with
`p.adjust < 0.05` already has `pvalue < 0.05`).

In [ ]:
library(here)
library(clusterProfiler)

## Load MSigDB gene sets

In [ ]:
msig_gmt <- clusterProfiler::read.gmt(here("data/pathways/msigdb.v2026.1.Hs.symbols.gmt"))
message(sprintf("MSigDB gene sets loaded: %d", length(unique(msig_gmt$term))))

## Paths

In [ ]:
seed_dir   <- here("output/01_model_building/04_archs4/06_bp_coverage_rshall",
                   "06_bp_coverage_hall_rs_100", "hall_coverage_rs100_seed_1")
z_path     <- file.path(seed_dir, "CLAMPbase", "Z.csv")
b_path     <- file.path(seed_dir, "CLAMPbase", "B.csv")

output_dir <- here("output/03_model_biology/00_archs4/08_saturation_study/00_bp_saturation_study_ora_analysis")
dir.create(file.path(output_dir, "CLAMPbase"), recursive = TRUE, showWarnings = FALSE)

rs_pct <- 100L
seed   <- 1L

## Run ORA (single-threaded, one model, seed1 only)

In [ ]:
Z              <- read.csv(z_path, row.names = 1, check.names = FALSE)
universe_genes <- rownames(Z)
n_top          <- ceiling(0.01 * nrow(Z))
n_lvs          <- ncol(Z)
k_val          <- n_lvs

cache_path <- file.path(output_dir, "CLAMPbase",
                        sprintf("rs%d_k%d_seed%d_msigdb.rds", rs_pct, k_val, seed))
message(sprintf("rs%d%% k%d seed%d -> %s", rs_pct, k_val, seed, cache_path))

if (file.exists(cache_path)) {
  message("Already cached, skipping: ", cache_path)
} else {
  term_overlap   <- tapply(msig_gmt$gene %in% universe_genes, msig_gmt$term, sum)
  n_total_msigdb <- sum(term_overlap >= 10L)

  top_genes_per_lv <- apply(Z, 2, function(lv) {
    universe_genes[order(lv, decreasing = TRUE)[seq_len(n_top)]]
  })
  rm(Z); gc()

  n_samples <- if (file.exists(b_path)) {
    ncol(read.csv(b_path, nrows = 0, check.names = FALSE))
  } else NA_integer_

  t0 <- proc.time()[[3]]
  ora_results <- lapply(seq_len(n_lvs), function(i) {
    if (i %% 100 == 0) message(sprintf("  LV %d/%d (%ds elapsed)", i, n_lvs, round(proc.time()[[3]] - t0)))
    genes <- top_genes_per_lv[, i]
    tryCatch(
      clusterProfiler::enricher(
        gene          = genes,
        universe      = universe_genes,
        TERM2GENE     = msig_gmt,
        pAdjustMethod = "BH",
        pvalueCutoff  = 0.05,
        qvalueCutoff  = 1,
        minGSSize     = 10,
        maxGSSize     = 50000
      ),
      error = function(e) NULL
    )
  })

  all_dfs <- Filter(Negate(is.null), lapply(ora_results, function(r) {
    if (is.null(r) || nrow(as.data.frame(r)) == 0) return(NULL)
    as.data.frame(r)
  }))

  if (length(all_dfs) == 0) stop("No ORA results returned for: ", z_path)

  combined <- do.call(rbind, all_dfs)

  res <- list(
    n_samples      = n_samples,
    n_lvs          = n_lvs,
    n_top_genes    = n_top,
    n_total_msigdb = n_total_msigdb,
    terms_padj     = tapply(combined$p.adjust, combined$ID, min),
    n_studies      = NA_integer_
  )

  message(sprintf("Done in %ds", round(proc.time()[[3]] - t0)))
  saveRDS(res, cache_path)
  message("Saved: ", cache_path)
}